# Guardrails
    Safety Mechanism
Guardrails are safety mechanisms that controls what goes into the ai agent and what comes out of the agent.

They sit around your agent pipeline and see that the agent:
-   only processes safe and appropriate inputs
-   only performs approved actions
-   only returns validated and compliant outputs


## Types of Guardrails Approach
There are different types of guardrails approach
### Deterministic approach: 

-  In this approach, the validation is rule based like reqEX and Keywords and also the llm cost is zero in this approach.
- Semantic understanding may break in this approach

### Model-based approach:

- In this approach, we will send the request or the response to any low cost llms to check whether it is safe to process this input or return the output gievn by the user.
- Semantics are perfectly and easily understood in this approach

#### Types of Model based approach 

we use the langchain as guardrails. These are applied as middlewares on our agent path
1. Before Agent Hook
    - This prevents maliciouos or out-of-scope requests from ever reaching our agent
        - when user submits prompt, the prompt is first directly to sent to smaller and faster llm and decision is taken and then sent to next layer.
2. PII Middleware
    - This Ensures the sensitive data is never reached to the LLM, that prevents it from being leaked or stored in model memory
        - This sits right after Before Agent Hook, it scans the prompt and looks for any sensitive info or passwords or any other and those are hashed or masked
        - This is rarely llm, mostly we use Deterministic middleware here.
3. After Agent Hook
    - Catches Hallucinations, formatting errors and catches any toxic outputs before reaching the user or next system
        - It compares the final output against the retrieved RAG context. If the output claims something not in the context, the hook rejects it.
4. Human-in-the-Loop Middleware
    - Provides absolute safety fro high stake actions by physically pausing the agent.
5. Layered Guardrails
    - It is the architectural philosophy of combining all the above. If a user tries a highly sophisticated prompt injection, the architecture relies on multiple safety nets:
        - The Before Hook might miss it.
        - The PII Middleware scrubs the credit card number anyway.
        - The agent gets confused and outputs a toxic response.
        - The After Hook catches the toxicity and blocks it.
        - If the agent somehow bypasses that and tries to delete the database, the HITL Middleware pauses the system and alerts you.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
GROQ_API_KEY=os.getenv('GROQ_API_KEY')
OPENAI_API_KEY=os.getenv('OPEN_API_KEY')
GOOGLE_API_KEY=os.getenv('GOOGLE_API_KEY')

## Deterministic Approach

In [2]:
def deterministic_approach(query:str)-> bool:
    banned_keywords=['malware','hack','exploit']
    return any(kw in query.lower() for kw in banned_keywords)

text_inputs=['how to hack database','what is the capital of france','i want to learn about malware spreading']

for text in text_inputs:
    if deterministic_approach(text):
        print('Blocked')
    else: print('Safe')

Blocked
Safe
Blocked


## Model Based Approach

In [3]:
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
groq_model=ChatGroq(model='qwen/qwen3-32b',api_key=GROQ_API_KEY)
openai_model=ChatOpenAI(model='gpt-4o-mini',api_key=OPENAI_API_KEY)
gemini_model=ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite', api_key=GOOGLE_API_KEY)

In [51]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

models = [
    ChatGoogleGenerativeAI(
        model="gemini-2.5-flash-lite",
        google_api_key=GOOGLE_API_KEY
    ),

    ChatGroq(
        model="qwen/qwen3-32b",
        api_key=GROQ_API_KEY
    ),

    ChatGroq(
        model="openai/gpt-oss-120b",
        api_key=GROQ_API_KEY
    )
]
def invoke_with_fallback(messages):
    last_error = None

    for model in models:
        try:
            return model.invoke(messages)

        except Exception as e:
            print(f"Failed: {model}")
            print(e)
            last_error = e

    raise last_error

In [ ]:
def model_based_approach(query:str)->str:
        model=ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')
        prompt=f'''Is the following input safe to process
            Reply with only unblocked or blocked,
            Rules that should be followed strictly
            1)I need a single word output
            Input text by user {query}'''
        response=model.invoke([{'role':'user','content':prompt}])
        return response.content
for text in text_inputs:
        print(model_based_approach(text))


blocked
unblocked
unblocked


In [7]:
from langchain.agents.middleware import PIIMiddleware
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def find_customer(query:str)-> str:
    '''used to find customer records'''
    return f'''customer found for query {query}'''

agent=create_agent(model='google_genai:gemini-2.5-flash-lite',
                   tools=[find_customer],
                   middleware=[
                       PIIMiddleware(
                           pii_type='email',
                           strategy='redact',
                           apply_to_input=True,
                       ),
                       PIIMiddleware(
                           pii_type='credit_card',
                           strategy='mask',
                           apply_to_input=True
                       ),
                       PIIMiddleware(
                           pii_type='ip',
                           strategy='hash',
                           apply_to_input=True
                       ),
                       PIIMiddleware(
                           pii_type='mac_address',
                           strategy='mask',
                           apply_to_input=True
                       )
                    #    ,
                    #    PIIMiddleware(
                    #     pii_type='api_key',
                    #     detector=r'sk-[a-zA-Z0-9]{32}',
                    #     strategy='block',
                    #     apply_to_input=True
                    #    )
                    # ,
                    #    PIIMiddleware(
                    #        pii_type='url',
                    #        strategy='block',
                    #        apply_to_input=True
                    #    )
                   ])

In [13]:
from pprint import pprint

response=agent.invoke({'messages':[{'role':'user','content':'''Name: John Doe
My email: john.doe@example.com

Phone: +1-555-123-4567

credit_card: 4111 1111 1111 1111

ip: 203.0.113.45

mac_address: 00:1A:2B:3C:4D:5E

url: https://www.mybank.com/login
                                    
api_key: sk-AbC123XyZ789LmNoPqRsTeuVwXyZ1234

Help me reset my account password and update my payment details.'''}]})
pprint(response)

{'messages': [HumanMessage(content='Name: John Doe\nMy email: [REDACTED_EMAIL]\n\nPhone: +1-555-123-4567\n\ncredit_card: **** **** **** 1111\n\nip: <ip_hash:eb4b2dd3>\n\nmac_address: **:**:**:**:**:5E\n\nurl: https://www.mybank.com/login\n\napi_key: sk-AbC123XyZ789LmNoPqRsTeuVwXyZ1234\n\nHelp me reset my account password and update my payment details.', additional_kwargs={}, response_metadata={}, id='bf3807a2-cb72-4006-a86b-26a28158267f'),
              AIMessage(content="I can help you with that. Could you please confirm the email address you'd like to use for the password reset?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ede7a-499d-72a2-8082-85199fb53a89-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 172, 'output_tokens': 25, 'total_tokens': 197, 'input_token_details': {'cache_read': 0}})]}


## Human In the Loop Middleware

In [58]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.agents.middleware.human_in_the_loop import HITLResponse,Decision
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.tools import tool

@tool
def search_web(query:str)->str:
    '''used to search internet'''
    return f'''the results for the given query {query}'''

@tool
def send_email(query:str)->str:
    '''used to send email'''
    return f'''email has been sent to the user {query}'''

@tool
def delete_records(query:str)->str:
    '''used to delete records'''
    return f'''the record {query} has been deleted'''

model=ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')
openai_model=ChatOpenAI(model='gpt-4o-mini')
groq_model=ChatGroq(model='qwen/qwen3-32b')

hitl_agent=create_agent(
    model=groq_model,
    tools=[search_web,send_email,delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "search_web":True,
                "send_email":True,
                "delete_records":True
            }
        )
    ],
    checkpointer=InMemorySaver()
)



### 1.Reject action

In [59]:
config1={'configurable':{'thread_id':'test5'}}

response=hitl_agent.invoke({'messages':[{'role':'user',
                "content": """
Send an email to 1uu@gmail.com.

Subject: Test

Body:
Hello from Nitin.
"""}]},config=config1)
pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'query': '{"to": '
                                                                           '"1uu@gmail.com", '
                                                                           '"subject": '
                                                                           '"Test", '
                                                                           '"body": '
                                                                           '"Hello '
                                                                           'from '
                                                                           'Nitin."}'},
                                                         'description': 'Tool '
                                                                        'execution '
                                                                        'requires '
                                                      

In [28]:
state=hitl_agent.get_state(config1)
result=state.interrupts[0]
pprint(result.value)

{'action_requests': [{'args': {'query': 'To: 1uu@gmail.com Subject: Test Body: '
                                        'Hello from Nitin.'},
                      'description': 'Tool execution requires approval\n'
                                     '\n'
                                     'Tool: send_email\n'
                                     "Args: {'query': 'To: 1uu@gmail.com "
                                     "Subject: Test Body: Hello from Nitin.'}",
                      'name': 'send_email'}],
 'review_configs': [{'action_name': 'send_email',
                     'allowed_decisions': ['approve',
                                           'edit',
                                           'reject',
                                           'respond']}]}


In [60]:
approved_result=hitl_agent.invoke(Command(resume= {'decisions': [{'type': 'reject'}]}),config=config1)
approved_result

{'messages': [HumanMessage(content='\nSend an email to 1uu@gmail.com.\n\nSubject: Test\n\nBody:\nHello from Nitin.\n', additional_kwargs={}, response_metadata={}, id='bab4597a-a2c1-4087-b8b7-c105e2f85987'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to 1uu@gmail.com with the subject "Test" and the body "Hello from Nitin." Let me check the available functions. There\'s a send_email function that takes a query parameter. I need to structure the arguments correctly. The parameters should include the recipient, subject, and body. Let me format the query as a JSON object with those fields. So the arguments for send_email would be {"to": "1uu@gmail.com", "subject": "Test", "body": "Hello from Nitin."}. That should do it.\n', 'tool_calls': [{'id': '8dt2fj1dq', 'function': {'arguments': '{"query":"{\\"to\\": \\"1uu@gmail.com\\", \\"subject\\": \\"Test\\", \\"body\\": \\"Hello from Nitin.\\"}"}', 'name': 'send_email'}, 'type': 'functio

### Edit action

In [61]:
config2={'configurable':{'thread_id':'test8'}}

response=hitl_agent.invoke({'messages':[{'role':'user',
                "content": """
Send an email to 1uu@gmail.com.

Subject: Test

Body:
Hello from Nitin.
"""}]},config=config2)
response

{'messages': [HumanMessage(content='\nSend an email to 1uu@gmail.com.\n\nSubject: Test\n\nBody:\nHello from Nitin.\n', additional_kwargs={}, response_metadata={}, id='ec04f831-6948-4eba-b89a-d7fef6ba891a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to 1uu@gmail.com with the subject "Test" and the body "Hello from Nitin." Let me check the available tools. There\'s a function called send_email. The parameters for send_email require a query, which is a string. Wait, the parameters are defined as {"properties": {"query": {"type": "string"}}, "required": ["query"], "type": "object"}. So the send_email function expects a query parameter that\'s a string. But the user provided specific details like recipient, subject, and body. How do I structure the query? Maybe the query should be a JSON object containing all those details. Let me format the email information into a query string. The arguments should include the recipient, subject

In [41]:
snapshot = hitl_agent.get_state(config2)
pprint(snapshot.interrupts[0])

Interrupt(value={'action_requests': [{'args': {'query': 'To: 1uu@gmail.com\n'
                                                        'Subject: Test\n'
                                                        'Body: Hello from '
                                                        'Nitin.'},
                                      'description': 'Tool execution requires '
                                                     'approval\n'
                                                     '\n'
                                                     'Tool: send_email\n'
                                                     "Args: {'query': 'To: "
                                                     '1uu@gmail.com\\nSubject: '
                                                     'Test\\nBody: Hello from '
                                                     "Nitin.'}",
                                      'name': 'send_email'}],
                 'review_configs': [{'action_name': 'send_emai

In [62]:
approved_result=hitl_agent.invoke(Command(resume={'decisions':[
    {'type':'edit',
     'edited_action':{
        'name':'send_email',
        'args':{
            'query':'nitin@gmail.com'}
            }
            }
            ]
            }
            ),config=config2)
approved_result

{'messages': [HumanMessage(content='\nSend an email to 1uu@gmail.com.\n\nSubject: Test\n\nBody:\nHello from Nitin.\n', additional_kwargs={}, response_metadata={}, id='ec04f831-6948-4eba-b89a-d7fef6ba891a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to 1uu@gmail.com with the subject "Test" and the body "Hello from Nitin." Let me check the available tools. There\'s a function called send_email. The parameters for send_email require a query, which is a string. Wait, the parameters are defined as {"properties": {"query": {"type": "string"}}, "required": ["query"], "type": "object"}. So the send_email function expects a query parameter that\'s a string. But the user provided specific details like recipient, subject, and body. How do I structure the query? Maybe the query should be a JSON object containing all those details. Let me format the email information into a query string. The arguments should include the recipient, subject

### Approve action

In [63]:
config3={'configurable':{'thread_id':'test01'}}

response=hitl_agent.invoke({'messages':[{'role':'user',
                "content": """
Send an email to 1uu@gmail.com.

Subject: Test

Body:
Hello from Nitin.
"""}]},config=config3)
response

{'messages': [HumanMessage(content='\nSend an email to 1uu@gmail.com.\n\nSubject: Test\n\nBody:\nHello from Nitin.\n', additional_kwargs={}, response_metadata={}, id='a487898e-ae0d-4646-b580-a5b2aace7c29'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to 1uu@gmail.com with the subject "Test" and the body "Hello from Nitin." Let me check the available functions.\n\nThe tools provided include send_email, which requires a query parameter. The query should probably contain the email details. Let me see the function signature again. The send_email function has parameters with a query string. So I need to structure the query to include the recipient, subject, and body.\n\nMaybe the query should be formatted in a way that specifies these fields. For example, "to:1uu@gmail.com subject:Test body:Hello from Nitin." The function might parse this query string to extract the necessary parts. I\'ll use the send_email function with this query 

In [ ]:
hitl_result=hitl_agent.invoke(Command(resume={'decisions':[{'type':'approve'}]}),config=config3)
pprint(hitl_result)

{'messages': [HumanMessage(content='\nSend an email to 1uu@gmail.com.\n\nSubject: Test\n\nBody:\nHello from Nitin.\n', additional_kwargs={}, response_metadata={}, id='a487898e-ae0d-4646-b580-a5b2aace7c29'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to 1uu@gmail.com with the subject "Test" and the body "Hello from Nitin." Let me check the available functions.\n\nThe tools provided include send_email, which requires a query parameter. The query should probably contain the email details. Let me see the function signature again. The send_email function has parameters with a query string. So I need to structure the query to include the recipient, subject, and body.\n\nMaybe the query should be formatted in a way that specifies these fields. For example, "to:1uu@gmail.com subject:Test body:Hello from Nitin." The function might parse this query string to extract the necessary parts. I\'ll use the send_email function with

: 